全连接层参数太多
卷积每次处理一张图用同一个kernel，参数少
RNN处理有序列关系的数据

In [1]:
import torch
batch_size = 1
seq_len = 3
input_size = 4
hidden_size = 2
cell = torch.nn.RNNCell(input_size=input_size, hidden_size=hidden_size)

# (seq, batch, features)
dataset = torch.randn(seq_len, batch_size, input_size)
hidden = torch.zeros(batch_size, hidden_size)

for idx, input in enumerate(dataset):
    print('=' * 20, idx, '=' * 20)
    print('Input size: ', input.shape)
    hidden = cell(input, hidden)
    print('outputs size: ', hidden.shape)
    print(hidden)

==================== 0 ====================
Input size:  torch.Size([1, 4])
outputs size:  torch.Size([1, 2])
tensor([[0.0880, 0.7886]], grad_fn=<TanhBackward0>)
==================== 1 ====================
Input size:  torch.Size([1, 4])
outputs size:  torch.Size([1, 2])
tensor([[0.1331, 0.9281]], grad_fn=<TanhBackward0>)
==================== 2 ====================
Input size:  torch.Size([1, 4])
outputs size:  torch.Size([1, 2])
tensor([[0.1220, 0.5516]], grad_fn=<TanhBackward0>)


In [2]:
import torch
batch_size = 1
seq_len = 3
input_size = 4
hidden_size = 2
num_layers = 1
cell = torch.nn.RNN(input_size=input_size, hidden_size=hidden_size,
num_layers=num_layers)

# (seqLen, batchSize, inputSize)
inputs = torch.randn(seq_len, batch_size, input_size)
hidden = torch.zeros(num_layers, batch_size, hidden_size)

out, hidden = cell(inputs, hidden)

print('Output size:', out.shape)
print('Output:', out)
print('Hidden size: ', hidden.shape)
print('Hidden: ', hidden)

Output size: torch.Size([3, 1, 2])
Output: tensor([[[ 0.4866, -0.6590]],

        [[ 0.8301,  0.3805]],

        [[ 0.6213,  0.2702]]], grad_fn=<StackBackward0>)
Hidden size:  torch.Size([1, 1, 2])
Hidden:  tensor([[[0.6213, 0.2702]]], grad_fn=<StackBackward0>)


In [ ]:
import torch
batch_size = 1
seq_len = 3
input_size = 4
hidden_size = 2
num_layers = 1
cell = torch.nn.RNN(input_size=input_size, hidden_size=hidden_size,
num_layers=num_layers, batch_first=True)


# (seqLen, batchSize, inputSize)
inputs = torch.randn(batch_size, seq_len, input_size)
hidden = torch.zeros(num_layers, batch_size, hidden_size)

out, hidden = cell(inputs, hidden)

print('Output size:', out.shape)
print('Output:', out)
print('Hidden size: ', hidden.shape)
print('Hidden: ', hidden)

RNNcell

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

input_size = 4
hidden_size = 4
batch_size = 1

idx2char = ['e', 'h', 'l', 'o']
x_data = [1, 0, 2, 2, 3]
y_data = [3, 1, 2, 3, 2]
one_hot_lookup=[[1, 0, 0, 0],
                [0, 1, 0, 0],
                [0, 0, 1, 0],
                [0, 0, 0, 1]]
x_one_hot = [one_hot_lookup[x] for x in x_data]
inputs = torch.Tensor(x_one_hot).view(-1, batch_size, input_size)
labels = torch.LongTensor(y_data).view(-1, 1)

class Model(torch.nn.Module):
    def __init__(self, input_size, hidden_size, batch_size):
        super(Model, self).__init__()
        self.batch_size = batch_size
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.rnncell = torch.nn.RNNCell(input_size=self.input_size,
                                        hidden_size=self.hidden_size)
        
    def forward(self, input, hidden):
        hidden = self.rnncell(input, hidden)
        return hidden
    
    def init_hidden(self):
        return torch.zeros(self.batch_size, self.hidden_size)
    
net = Model(input_size, hidden_size, batch_size)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)

for epoch in range(15):
    loss = 0
    optimizer.zero_grad()
    hidden = net.init_hidden()
    print('Predicted string: ', end='')
    for input, label in zip(inputs, labels):
        hidden = net(input, hidden)
        loss += criterion(hidden, label)
        _, idx = hidden.max(dim=1)
        print(idx2char[idx.item()], end='')
    loss.backward()
    optimizer.step()
    print(', Epoch [%d/15] loss=%.4f' % (epoch+1, loss.item()))


Predicted string: heeee, Epoch [1/15] loss=8.2089
Predicted string: hhhll, Epoch [2/15] loss=6.5162
Predicted string: hhlhl, Epoch [3/15] loss=5.5632
Predicted string: ohlll, Epoch [4/15] loss=4.9370
Predicted string: ohlll, Epoch [5/15] loss=4.4107
Predicted string: ohlll, Epoch [6/15] loss=3.9878
Predicted string: ohlol, Epoch [7/15] loss=3.6789
Predicted string: ohlol, Epoch [8/15] loss=3.4685
Predicted string: ohlol, Epoch [9/15] loss=3.2908
Predicted string: ohlol, Epoch [10/15] loss=3.1053
Predicted string: ohlol, Epoch [11/15] loss=2.9386
Predicted string: ohlol, Epoch [12/15] loss=2.8197
Predicted string: ohlol, Epoch [13/15] loss=2.7419
Predicted string: ohlol, Epoch [14/15] loss=2.6782
Predicted string: ohlol, Epoch [15/15] loss=2.6115


RNN

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim

input_size = 4
hidden_size = 4
num_layers = 1
batch_size = 1
seq_len = 5

idx2char = ['e', 'h', 'l', 'o']
x_data = [1, 0, 2, 2, 3]
y_data = [3, 1, 2, 3, 2]
one_hot_lookup = [[1, 0, 0, 0],
                [0, 1, 0, 0],
                [0, 0, 1, 0],
                [0, 0, 0, 1]]
x_one_hot = [one_hot_lookup[x] for x in x_data]
inputs = torch.Tensor(x_one_hot).view(seq_len, batch_size, input_size)
labels = torch.LongTensor(y_data)

class Model(torch.nn.Module):
    def __init__(self, input_size, hidden_size, batch_size, num_layers=1):
        super(Model, self).__init__()
        self.num_layers = num_layers
        self.batch_size = batch_size
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.rnn = torch.nn.RNN(input_size=self.input_size,
                                hidden_size=self.hidden_size,
                                num_layers=num_layers)
    def forward(self, input):
        hidden = torch.zeros(self.num_layers,
                            self.batch_size,
                            self.hidden_size)
        out, _ = self.rnn(input, hidden)
        return out.view(-1, self.hidden_size)
    
net = Model(input_size, hidden_size, batch_size, num_layers)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)

for epoch in range(15):
    #Training Steps
    optimizer.zero_grad()
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    _, idx = outputs.max(dim=1)
    idx = idx.data.numpy()
    print('Predicted: ', ''.join([idx2char[x] for x in idx]), end='')
    print(', Epoch [%d/15] loss = %.3f' % (epoch + 1, loss.item()))


Predicted:  elool, Epoch [1/15] loss = 1.282
Predicted:  lllol, Epoch [2/15] loss = 1.104
Predicted:  ollol, Epoch [3/15] loss = 1.016
Predicted:  ollol, Epoch [4/15] loss = 0.954
Predicted:  ollol, Epoch [5/15] loss = 0.899
Predicted:  ollll, Epoch [6/15] loss = 0.843
Predicted:  ohlll, Epoch [7/15] loss = 0.789
Predicted:  ohlll, Epoch [8/15] loss = 0.739
Predicted:  ohlll, Epoch [9/15] loss = 0.699
Predicted:  ohlll, Epoch [10/15] loss = 0.674
Predicted:  ohlll, Epoch [11/15] loss = 0.653
Predicted:  ohlll, Epoch [12/15] loss = 0.627
Predicted:  ohlol, Epoch [13/15] loss = 0.599
Predicted:  ohlol, Epoch [14/15] loss = 0.568
Predicted:  ohlol, Epoch [15/15] loss = 0.529


Embedding

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

num_class = 4
input_size = 4
hidden_size = 8
embedding_size = 10
num_layers = 2
batch_size = 1
seq_len = 5

idx2char = ['e', 'h', 'l', 'o']
x_data = [[1, 0, 2, 2, 3]] # (batch, seq_len)
y_data = [3, 1, 2, 3, 2] # (batch * seq_len)
inputs = torch.LongTensor(x_data)
labels = torch.LongTensor(y_data)

class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.emb = torch.nn.Embedding(input_size, embedding_size)
        self.rnn = torch.nn.RNN(input_size=embedding_size,
                                hidden_size=hidden_size,
                                num_layers=num_layers,
                                batch_first=True) 
        self.fc = torch.nn.Linear(hidden_size, num_class)
    def forward(self, x):
        hidden = torch.zeros(num_layers, x.size(0), hidden_size)
        x = self.emb(x) # (batch, seqLen, embeddingSize)
        x, _ = self.rnn(x, hidden)
        x = self.fc(x)
        return x.view(-1, num_class)
    
net = Model()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.05)

for epoch in range(15):
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, idx = outputs.max(dim=1)
        idx = idx.data.numpy()
        print('Predicted: ', ''.join([idx2char[x] for x in idx]), end='')
        print(', Epoch [%d/15] loss = %.3f' % (epoch + 1, loss.item()))


Predicted:  eheee, Epoch [1/15] loss = 1.472
Predicted:  hhehl, Epoch [2/15] loss = 1.169
Predicted:  hhlhl, Epoch [3/15] loss = 0.920
Predicted:  ohlol, Epoch [4/15] loss = 0.711
Predicted:  ohlol, Epoch [5/15] loss = 0.540
Predicted:  ohlol, Epoch [6/15] loss = 0.392
Predicted:  ohlol, Epoch [7/15] loss = 0.280
Predicted:  ohlol, Epoch [8/15] loss = 0.207
Predicted:  ohlol, Epoch [9/15] loss = 0.158
Predicted:  ohlol, Epoch [10/15] loss = 0.123
Predicted:  ohlol, Epoch [11/15] loss = 0.097
Predicted:  ohlol, Epoch [12/15] loss = 0.076
Predicted:  ohlol, Epoch [13/15] loss = 0.061
Predicted:  ohlol, Epoch [14/15] loss = 0.049
Predicted:  ohlol, Epoch [15/15] loss = 0.040


# Comparing the three implementations

All three solve the **same toy task**: learn the character mapping `"hello"` → `"ohlol"` (a sequence-labeling problem where every input character has a target output character). They differ only in *how* the recurrent computation is wired and fed.

Vocabulary: `idx2char = ['e', 'h', 'l', 'o']` (4 distinct characters).
Input `"hello"` = indices `[1, 0, 2, 2, 3]`; target `"ohlol"` = indices `[3, 1, 2, 3, 2]`.

The core recurrence is identical in every version:

$$h_t = \tanh(W_{ih}\,x_t + b_{ih} + W_{hh}\,h_{t-1} + b_{hh})$$

---

## 1) `RNNCell` — the manual loop

```python
self.rnncell = torch.nn.RNNCell(input_size=4, hidden_size=4)
# training loop:
hidden = net.init_hidden()              # (batch, hidden) = (1, 4)
for input, label in zip(inputs, labels):
    hidden = net(input, hidden)         # ONE timestep per call
    loss += criterion(hidden, label)
```

- **`RNNCell` is a single timestep.** It takes `x_t` of shape `(batch, input_size)` and `h_{t-1}` of shape `(batch, hidden_size)`, returns `h_t` of shape `(batch, hidden_size)`. **You** write the Python `for` loop over the sequence.
- The hidden state is a **2-D** tensor `(batch, hidden_size)` — no layer/sequence dimension, because you handle those yourself.
- Here `hidden_size == num_class == 4`, so the hidden vector **is** the class logits — there is no separate output layer.
- Loss is **accumulated (summed)** across the 5 timesteps before one `backward()`.
- **Pedagogically clearest** (you see every step), but the Python loop is slow for long sequences.

## 2) `nn.RNN` — the whole sequence at once

```python
self.rnn = torch.nn.RNN(input_size=4, hidden_size=4, num_layers=1)
def forward(self, input):                       # input: (seq, batch, input_size)
    hidden = torch.zeros(num_layers, batch, hidden_size)   # 3-D!
    out, _ = self.rnn(input, hidden)            # out: (seq, batch, hidden)
    return out.view(-1, hidden_size)            # (seq*batch, hidden)
```

- **`nn.RNN` runs the entire loop internally** (optimized C++/CUDA), so you pass the full sequence in one call and get every timestep's output back. Same math as version 1, just faster and less code.
- The initial hidden state is **3-D**: `(num_layers, batch, hidden_size)`. This 3-D requirement is exactly what produced the earlier `Expected hidden size` error when the dimension ordering didn't match the data layout.
- `out` holds the hidden state at **every** timestep, shape `(seq, batch, hidden)`. Reshaping to `(seq*batch, hidden)` lines it up against the flat `(seq,)` label tensor for `CrossEntropyLoss`.
- Again `hidden_size == num_class`, so the output is used directly as logits.
- Loss is a **single** `criterion(outputs, labels)` call → averaged (mean) over the 5 timesteps.

## 3) `Embedding` + stacked `nn.RNN` + `Linear` — the realistic pattern

```python
self.emb = nn.Embedding(vocab_size=4, embedding_size=10)
self.rnn = nn.RNN(input_size=10, hidden_size=8, num_layers=2, batch_first=True)
self.fc  = nn.Linear(hidden_size=8, num_class=4)
def forward(self, x):                  # x: (batch, seq) of integer indices
    hidden = torch.zeros(num_layers, x.size(0), hidden_size)
    x = self.emb(x)                    # (batch, seq, 10)
    x, _ = self.rnn(x, hidden)         # (batch, seq, 8)
    x = self.fc(x)                     # (batch, seq, 4)
    return x.view(-1, num_class)       # (seq, 4)
```

Three things change versus versions 1–2:

1. **Input is integer indices, not one-hot.** `nn.Embedding` is a *learnable* lookup table (`vocab_size × embedding_size`) that replaces the fixed one-hot encoding with a trained dense vector. This is the standard approach in real NLP — one-hot doesn't scale to large vocabularies and carries no notion of similarity.
2. **`hidden_size (8) ≠ num_class (4)`**, so a `Linear` head projects each timestep's hidden vector down to the 4 class logits. Decoupling hidden width from output width is what you do in any real model.
3. **`num_layers=2`** stacks two RNNs (a *deep* RNN): the first layer's per-step output feeds the second layer as input. **`batch_first=True`** makes the layout `(batch, seq, feature)` to match the embedding output.

Because of the embedding + deep RNN + linear head, this model has many more parameters (see the comparison cell) and is the template you'd scale up for actual sequence modeling — which is also why it converged fastest here (loss → ~0.04).

---

### At a glance

| | 1) RNNCell loop | 2) nn.RNN | 3) Embedding+RNN |
|---|---|---|---|
| Input encoding | one-hot | one-hot | learned embedding (indices) |
| Sequence loop | manual Python `for` | inside `nn.RNN` | inside `nn.RNN` |
| Hidden tensor rank | 2-D `(B, H)` | 3-D `(L, B, H)` | 3-D `(L, B, H)` |
| Stacked layers | 1 | 1 | 2 |
| Output → logits | hidden used directly | hidden used directly | separate `Linear` |
| Loss reduction | **sum** over steps | **mean** over steps | **mean** over steps |
| Data layout | `(seq, B, feat)` | `(seq, B, feat)` | `(B, seq, feat)` (`batch_first`) |

> The **sum vs mean** row also explains the convergence-speed question from earlier: the `RNNCell` loss starts ~5× larger, so for the same per-step error its effective gradient step is larger.

The next cell quantifies these differences (parameter counts + a shape trace); the cell after that visualizes the shared unrolled-RNN computation.

In [ ]:
# Side-by-side comparison of the three implementations:
# parameter counts, layer structure, and the shape trace through the Embedding model.
import torch


class CellModel(torch.nn.Module):
    """1) manual per-timestep loop with RNNCell."""
    def __init__(self, input_size, hidden_size, batch_size):
        super().__init__()
        self.batch_size, self.hidden_size = batch_size, hidden_size
        self.rnncell = torch.nn.RNNCell(input_size=input_size, hidden_size=hidden_size)
    def forward(self, input, hidden):
        return self.rnncell(input, hidden)
    def init_hidden(self):
        return torch.zeros(self.batch_size, self.hidden_size)


class RNNModel(torch.nn.Module):
    """2) whole sequence through nn.RNN."""
    def __init__(self, input_size, hidden_size, batch_size, num_layers=1):
        super().__init__()
        self.num_layers, self.batch_size, self.hidden_size = num_layers, batch_size, hidden_size
        self.rnn = torch.nn.RNN(input_size=input_size, hidden_size=hidden_size,
                                num_layers=num_layers)
    def forward(self, input):
        hidden = torch.zeros(self.num_layers, self.batch_size, self.hidden_size)
        out, _ = self.rnn(input, hidden)
        return out.view(-1, self.hidden_size)


class EmbModel(torch.nn.Module):
    """3) Embedding + stacked RNN + Linear head."""
    def __init__(self, vocab_size, embedding_size, hidden_size, num_class, num_layers):
        super().__init__()
        self.num_layers, self.hidden_size = num_layers, hidden_size
        self.emb = torch.nn.Embedding(vocab_size, embedding_size)
        self.rnn = torch.nn.RNN(input_size=embedding_size, hidden_size=hidden_size,
                                num_layers=num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_size, num_class)
    def forward(self, x):
        hidden = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        x, _ = self.rnn(self.emb(x), hidden)
        return self.fc(x).view(-1, self.fc.out_features)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


m1 = CellModel(input_size=4, hidden_size=4, batch_size=1)
m2 = RNNModel(input_size=4, hidden_size=4, batch_size=1, num_layers=1)
m3 = EmbModel(vocab_size=4, embedding_size=10, hidden_size=8, num_class=4, num_layers=2)

rows = [
    ("Feature", "1) RNNCell loop", "2) nn.RNN", "3) Embedding+RNN"),
    ("Input to network", "one-hot (seq,B,4)", "one-hot (seq,B,4)", "indices (B,seq)"),
    ("Sequence handling", "Python for-loop", "single RNN call", "single RNN call"),
    ("Hidden tensor rank", "2-D (B,H)", "3-D (L,B,H)", "3-D (L,B,H)"),
    ("# stacked layers", "1", "1", "2"),
    ("Output head", "hidden = logits", "hidden = logits", "Linear 8->4"),
    ("Loss reduction", "sum over steps", "mean over steps", "mean over steps"),
    ("Total parameters", str(count_params(m1)), str(count_params(m2)), str(count_params(m3))),
]
w = [max(len(r[c]) for r in rows) for c in range(4)]
for i, r in enumerate(rows):
    print(" | ".join(r[c].ljust(w[c]) for c in range(4)))
    if i == 0:
        print("-+-".join("-" * w[c] for c in range(4)))

print("\nWhy model 3 has many more parameters -- per-module breakdown:")
for name, p in m3.named_parameters():
    print(f"  {name:18s} {str(tuple(p.shape)):10s} -> {p.numel():>3d}")

print("\nShape trace through model 3's forward():")
x = torch.LongTensor([[1, 0, 2, 2, 3]])      # (batch=1, seq=5) integer indices
e = m3.emb(x)
o, hn = m3.rnn(e, torch.zeros(2, x.size(0), 8))
print(f"  indices            {tuple(x.shape)}")
print(f"  after Embedding    {tuple(e.shape)}   (batch, seq, embedding=10)")
print(f"  RNN output         {tuple(o.shape)}    (batch, seq, hidden=8)")
print(f"  RNN h_n            {tuple(hn.shape)}    (num_layers=2, batch, hidden=8)")
print(f"  after Linear       {tuple(m3.fc(o).shape)}    (batch, seq, num_class=4)")
print(f"  view(-1, 4)        {tuple(m3.fc(o).view(-1, 4).shape)}       (seq*batch, num_class) -> matches labels")


In [ ]:
# Visualization: how an RNN unrolls over the sequence
# (Same picture applies to all three implementations -- they share the recurrence.)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # avoid torch/matplotlib OpenMP clash on Anaconda
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

idx2char = ['e', 'h', 'l', 'o']
x_data = [1, 0, 2, 2, 3]   # "hello"
y_data = [3, 1, 2, 3, 2]   # "ohlol"
seq_len = len(x_data)

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(-0.6, seq_len + 0.6)
ax.set_ylim(-0.5, 4.2)
ax.axis('off')

cell_w, cell_h = 0.7, 0.7
y_cell = 1.6

# initial hidden state feeding the first cell
ax.annotate('', xy=(0.5 - cell_w / 2, y_cell + cell_h / 2),
            xytext=(-0.3, y_cell + cell_h / 2),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(-0.4, y_cell + cell_h / 2 + 0.25, '$h_0$\n(zeros)',
        ha='center', va='bottom', fontsize=10, color='gray')

for t in range(seq_len):
    cx = t + 0.5
    # the RNN cell at timestep t
    ax.add_patch(Rectangle((cx - cell_w / 2, y_cell), cell_w, cell_h,
                           facecolor='#cfe8ff', edgecolor='#1f6fb2', lw=2, zorder=2))
    ax.text(cx, y_cell + cell_h / 2, 'RNN\nCell', ha='center', va='center',
            fontsize=10, fontweight='bold', zorder=3)

    # input x_t (green, from below)
    ax.text(cx, 0.55, f'x{t} = "{idx2char[x_data[t]]}"', ha='center', va='center', fontsize=10)
    ax.annotate('', xy=(cx, y_cell), xytext=(cx, 0.8),
                arrowprops=dict(arrowstyle='->', color='#2ca02c', lw=2))

    # recurrent hidden state (red, horizontal) -> next cell
    if t < seq_len - 1:
        ax.annotate('', xy=(cx + 1 - cell_w / 2, y_cell + cell_h / 2),
                    xytext=(cx + cell_w / 2, y_cell + cell_h / 2),
                    arrowprops=dict(arrowstyle='->', color='#d62728', lw=2))
        ax.text(cx + 0.5, y_cell + cell_h / 2 + 0.12, f'h{t+1}',
                ha='center', va='bottom', fontsize=9, color='#d62728')

    # output / target y_t (purple, upward)
    ax.annotate('', xy=(cx, 3.0), xytext=(cx, y_cell + cell_h),
                arrowprops=dict(arrowstyle='->', color='#9467bd', lw=2))
    ax.text(cx, 3.25, f'y{t}\n"{idx2char[y_data[t]]}"', ha='center', va='center',
            fontsize=10, color='#9467bd', fontweight='bold')

# final hidden state leaving the last cell
last_cx = seq_len - 1 + 0.5
ax.annotate('', xy=(last_cx + 0.7, y_cell + cell_h / 2),
            xytext=(last_cx + cell_w / 2, y_cell + cell_h / 2),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(last_cx + 0.8, y_cell + cell_h / 2 + 0.25, f'h{seq_len}',
        ha='left', va='bottom', fontsize=10, color='gray')

ax.text(seq_len / 2, 4.0, 'Unrolled RNN over the sequence  "hello" -> "ohlol"',
        ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(seq_len / 2, -0.35,
        r'$h_t = \tanh(W_{ih}\,x_t + b_{ih} + W_{hh}\,h_{t-1} + b_{hh})$',
        ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

# How each implementation realizes this picture:
#   1) RNNCell : YOU write the for-loop -> one blue box per iteration (h is 2-D (B,H))
#   2) nn.RNN  : the loop happens inside nn.RNN in one call (h is 3-D (L,B,H))
#   3) Emb+RNN : same loop, but x_t is a learned embedding vector and the purple
#                output passes through a Linear head; 2 stacked rows of blue boxes.
